# Market access for Interwar Poland districts (1913, 1924, 1938)

This notebook computes **district-level market access** for each year using:

- a **routable travel-time matrix** (minutes) between *points* (district centroids + cities)  
  (`distance_matrix_long_1913.csv`, `..._1924.csv`, `..._1938.csv`)
- **district rural population** (assigned to the district centroid)
- **city population** (assigned to city centroids, spatially joined to districts)

## Key idea: collapse point-to-point travel times into district-to-district travel times

Instead of computing market access at each point and then averaging within districts, we first construct a
**district-to-district travel-time matrix** by combining all point-to-point connections between two districts.

For two districts $i$ and $j$, let $p \in i$ and $q \in j$ index all points in those districts:

- district centroid point (carrying **rural** population),
- all city points (carrying **city** population).

Define each point’s population mass as $Pop_p$. We compute the
**population-pair-weighted mean travel time** as:

$$
\tilde{t}_{ij} =
\frac{\sum_{p \in i}\sum_{q \in j} Pop_p \, Pop_q \, t_{pq}}
{\sum_{p \in i}\sum_{q \in j} Pop_p \, Pop_q},
\qquad i \neq j
$$

This yields one effective travel time between every ordered pair of districts.

## District market access

Once $\tilde{t}_{ij}$ is constructed, we compute market access using **total district population**:

$$
Pop^{district}_j = Rural_j + \sum_{c \in j} Pop_c
$$

Districts are then treated as the nodes in the gravity calculation.

The output is one row per district per year with the district market access level and the underlying population components.


## Market access functional form

After constructing the district-to-district travel-time matrix $\tilde{t}_{ij}$,
we compute a gravity-style market access measure at the **district** level:

$$
MA_i = \sum_{j \neq i}
\frac{Pop^{district}_j}{(\tilde{t}_{ij} + \varepsilon)^{\theta}}
$$

where:
- $\tilde{t}_{ij}$ is the **population-pair-weighted mean** travel time (minutes) between districts $i$ and $j$
- $Pop^{district}_j$ is the total population in district $j$ (rural + all cities)
- $\theta$ controls decay with travel time (default **1.0**)
- $\varepsilon$ avoids division by zero (default **1 minute**)

Optionally, we can use an exponential decay form:

$$
MA_i = \sum_{j \neq i} Pop^{district}_j \cdot e^{-\alpha \tilde{t}_{ij}}
$$

Both functional forms are implemented below.

## Population for 1924
- City population in **1924** is **linearly interpolated** between **1921** and **1931**.
- Rural population for **1924** is taken directly from `rural_population.csv`.

## Population for 1913 and 1938 (data gap handling)
Your `rural_population.csv` covers **1921–1938** (no 1913, no 1938).  
Your `city_population.csv` covers **1921, 1931, 1938** (no 1913).

To compute 1913 and 1938 market access, this notebook uses simple, explicit rules:
- **Rural 1913**: linear **extrapolation backwards** using 1921→1922 slope (can be changed to “use 1921 as 1913”).
- **Rural 1938**: linear **extrapolation forward** using 1937→1938 slope (or use 1938 as 1938).
- **City 1913**: linear **extrapolation backwards** from 1921→1931 trend (or use 1921 as 1913).

All these choices are parameterized and logged to allow for robustness checks.

In [1]:
import os
import sys
from pathlib import Path

os.chdir("../../..")
sys.path.append(str(Path.cwd() / "examples" / "interwar_poland"))

from global_definitions import (
    administrative_history,
    adm_history_plotter
)

from global_definitions import (
    d_city_mapping
)

os.chdir(str(Path.cwd() / "examples" / "interwar_poland" / "market_access"))

Loading changes list...
✅ Loaded 309 validated changes in 0.07 seconds.
Loading initial state...
✅ Loaded initial state.
Loading initial district registry...
✅ Loaded 292 validated districts in 0.10 seconds. Set their initial state timespans to (1921-02-19, 1939-09-01).
Loading initial region registry...
✅ Loaded 19 validated regions in 0.00 seconds. Set their initial state timespans to (1921-02-19, 1939-09-01)
Creating administrative history (sequentially applying changes)...
✅ Successfully applied all changes in 11.93 seconds. Administrative history database created.
Loading territories...
Loaded: districts_1922_generalized_dp_800m.shp (271 rows)
Loaded: districts_1929_generalized_dp_800m.shp (281 rows)
Loaded: districts_1931_generalized_dp_800m.shp (283 rows)
Loaded: districts_1934_generalized_dp_800m.shp (264 rows)
Loaded: districts_1939_generalized_dp_800m.shp (265 rows)
✅ Successfully loaded all territories in 1.16 seconds.
Deducing all possible dist territories on the basis of t

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
from datetime import datetime

# ---------- Paths (edit if needed) ----------
DATA_DIR = Path("data")  # folder containing the csvs/geojson
DISTRICTS_GEOJSON = DATA_DIR / "districts_1934_10_1.geojson"

CITY_POP_CSV = DATA_DIR / "city_population.csv"
RURAL_POP_CSV = DATA_DIR / "rural_population.csv"

DM_1913 = DATA_DIR / "distance_matrix_long_1913_baseline.csv"
DM_1924 = DATA_DIR / "distance_matrix_long_1924_baseline.csv"
DM_1938 = DATA_DIR / "distance_matrix_long_1938_baseline.csv"

DM_HORSE_KM_1913 = DATA_DIR / "distance_matrix_horse_km_long_1913_baseline.csv"
DM_HORSE_KM_1924 = DATA_DIR / "distance_matrix_horse_km_long_1924_baseline.csv"
DM_HORSE_KM_1938 = DATA_DIR / "distance_matrix_horse_km_long_1938_baseline.csv"

DM_RAIL_KM_1913 = DATA_DIR / "distance_matrix_rail_km_long_1913_baseline.csv"
DM_RAIL_KM_1924 = DATA_DIR / "distance_matrix_rail_km_long_1924_baseline.csv"
DM_RAIL_KM_1938 = DATA_DIR / "distance_matrix_rail_km_long_1938_baseline.csv"

MA_PLOTS_DIR = Path("./plots/market_access")
MA_PLOTS_DIR.mkdir(exist_ok=True)

BORDER_CROSSING_PLOTS_DIR = Path("./plots/border_crossing")
BORDER_CROSSING_PLOTS_DIR.mkdir(exist_ok=True)

ADM_STATE_DATE = datetime(1934, 10, 1)

# City coords GeoJSON:
HERE = Path.cwd()

CITY_COORDS_GEOJSON = (
    HERE / "../../../data/adm_histories/interwar_poland/cities_coords/cities_coords.geojson"
).resolve()

# ---------- Market access parameters ----------
# Power decay: MA_p = sum Pop_q / (time_min + eps)^theta
THETA = 1.0
EPS_MIN = 1.0

# Optional exponential decay alternative:
USE_EXPONENTIAL = False
ALPHA_PER_MIN = 0.01  # only used if USE_EXPONENTIAL=True; MA_p = sum Pop_q * exp(-alpha * time)

# ---------- Population gap-handling parameters ----------
CITY_1913_METHOD = "extrapolate"   # "extrapolate" or "use_1921"
RURAL_1913_METHOD = "extrapolate"  # "extrapolate" or "use_1921"
RURAL_1938_METHOD = "extrapolate"  # "extrapolate" or "use_1938"

print("Using city coords file:", CITY_COORDS_GEOJSON)
assert CITY_COORDS_GEOJSON.exists()


Using city coords file: E:\git_projects\border_harmonization_toolkit\data\adm_histories\interwar_poland\cities_coords\cities_coords.geojson


## 1) Load districts and city coordinates, assign cities to districts

We assign each city to a district polygon using a spatial join (point-in-polygon).
If your district file is in a projected CRS, we reproject to EPSG:4326.


In [3]:
# Load districts polygons
districts = gpd.read_file(DISTRICTS_GEOJSON)
assert "District" in districts.columns, "Expected 'District' column in districts geojson"

# Ensure we have a CRS and work in WGS84 for joining with lon/lat city coords
if districts.crs is None:
    # If CRS is missing, we assume WGS84; adjust if needed.
    districts = districts.set_crs("EPSG:4326")
districts_wgs84 = districts.to_crs("EPSG:4326")

# Load city coordinates from GeoJSON
cities_gdf = gpd.read_file(CITY_COORDS_GEOJSON)

if "City" not in cities_gdf.columns:
    raise RuntimeError("cities_coords.geojson must have column 'City'")

# Ensure CRS
if cities_gdf.crs is None:
    raise RuntimeError("cities_coords.geojson has no CRS defined")

cities_gdf = cities_gdf.to_crs("EPSG:4326")

cities_gdf["City"] = cities_gdf["City"].astype(str).str.strip()

# Spatial join: city -> district
cities_with_district = gpd.sjoin(
    cities_gdf[["City", "geometry"]],
    districts_wgs84[["District", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"])

missing = cities_with_district["District"].isna().sum()
print("Cities with missing district assignment:", int(missing))

cities_with_district.head()

# Spatial join: city -> district
cities_with_district = gpd.sjoin(
    cities_gdf, districts_wgs84[["District", "geometry"]],
    how="left", predicate="within"
).drop(columns=["index_right"])

missing = cities_with_district["District"].isna().sum()
print("Cities with missing district assignment:", int(missing))
cities_with_district.head()


Cities with missing district assignment: 6
Cities with missing district assignment: 6


,City,Region,Comment,Wiki_link,geometry,District
0,Aleksandrów (Łódzki),Łódzkie,None,https://pl.wikipedia.org/wiki/Aleksandrów_Łódzki,POINT (19.30444 51.81944),ŁÓDZKI
1,Aleksandrów Kujawski,Warszawskie,None,https://pl.wikipedia.org/wiki/Aleksandrów_Kuja...,POINT (18.69333 52.87639),NIESZAWSKI
2,Andrychów,Krakowskie,None,https://pl.wikipedia.org/wiki/Andrychów,POINT (19.34111 49.855),WADOWICKI
3,Augustów,Białostockie,None,https://pl.wikipedia.org/wiki/Augustów,POINT (22.9775 53.84361),AUGUSTOWSKI
4,Baranowicze,Nowogródzkie,None,https://pl.wikipedia.org/wiki/Baranowicze,POINT (25.98333 53.11667),BARANOWICKI


## 2) Load populations and build population series for 1913, 1924, 1938

### Cities
- 1924: interpolate between 1921 and 1931
- 1913: extrapolate backwards from 1921→1931 trend (or use 1921)
- 1938: use provided 1938

### Rural (district)
- 1924: use provided 1924
- 1913: extrapolate backwards from 1921→1922 slope (or use 1921)
- 1938: extrapolate forward from 1937→1938 slope (or use 1938)

We then build a unified `pop_points` table with point IDs matching the distance matrices:
- `District:<District>` population = rural population
- `City:<City>` population = city population (only if city assigned to a district)


In [ ]:
# --- Load city population ---
city_pop = pd.read_csv(CITY_POP_CSV, sep=";", encoding="utf-8")
city_pop["City"] = city_pop["City"].astype(str).str.strip()

# Ensure numeric
for y in ["1921", "1931", "1938"]:
    city_pop[y] = pd.to_numeric(city_pop[y], errors="coerce")

# Merge with district assignment (so we can later aggregate by district)
city_pop = city_pop.merge(
    cities_with_district[["City", "District"]],
    on="City", how="left"
)

# Drop cities with no district assignment (you can keep them if you want cross-border cities as markets)
city_pop_assigned = city_pop.dropna(subset=["District"]).copy()

# City 1924 interpolation
city_pop_assigned["1924"] = city_pop_assigned["1921"] + (1924 - 1921) / (1931 - 1921) * (city_pop_assigned["1931"] - city_pop_assigned["1921"])

# City 1913 (method choice)
if CITY_1913_METHOD == "use_1921":
    city_pop_assigned["1913"] = city_pop_assigned["1921"]
elif CITY_1913_METHOD == "extrapolate":
    # linear trend from 1921 to 1931, extrapolate back to 1913
    slope = (city_pop_assigned["1931"] - city_pop_assigned["1921"]) / (1931 - 1921)
    city_pop_assigned["1913"] = city_pop_assigned["1921"] + (1913 - 1921) * slope
else:
    raise ValueError("CITY_1913_METHOD must be 'extrapolate' or 'use_1921'")

# --- Load rural population ---
rural = pd.read_csv(RURAL_POP_CSV, sep=";")
rural["District"] = rural["District"].astype(str).str.strip()

# Convert all year columns to numeric
year_cols = [c for c in rural.columns if c != "District"]
for c in year_cols:
    rural[c] = pd.to_numeric(rural[c], errors="coerce")

# Rural 1913
if RURAL_1913_METHOD == "use_1921":
    rural["1913"] = rural["1921"]
elif RURAL_1913_METHOD == "extrapolate":
    slope = rural["1922"] - rural["1921"]  # per year
    rural["1913"] = rural["1921"] + (1913 - 1921) * slope
else:
    raise ValueError("RURAL_1913_METHOD must be 'extrapolate' or 'use_1921'")

# Rural 1938
if RURAL_1938_METHOD == "use_1938":
    rural["1938"] = rural["1938"]
elif RURAL_1938_METHOD == "extrapolate":
    slope = rural["1938"] - rural["1937"]
    rural["1938"] = rural["1938"] + (1938 - 1938) * slope
else:
    raise ValueError("RURAL_1938_METHOD must be 'extrapolate' or 'use_1938'")

# Keep only needed years
rural_keep = rural[["District", "1913", "1924", "1938"]].copy()

city_keep = city_pop_assigned[["City", "District", "1913", "1924", "1938"]].copy()

print("City rows (assigned):", len(city_keep))
print("Rural rows:", len(rural_keep))
city_keep.head()


City rows (assigned): 650
Rural rows: 247


,City,District,1913,1924,1939
0,Aleksandrów (Łódzki),ŁÓDZKI,5581.6,9231.4,13333.0
1,Aleksandrów Kujawski,NIESZAWSKI,7095.4,8604.6,9554.0
2,Andrychów,WADOWICKI,2949.4,4629.1,7000.0
3,Augustów,AUGUSTOWSKI,6054.0,9777.5,17861.0
4,Baranowicze,BARANOWICKI,2333.4,14897.6,30119.0


## 3) Load distance matrices and compute **district-to-district** distances (population-weighted over points)

The time distance matrices are long-form with columns:
- `origin_id`, `origin_type`, `dest_id`, `dest_type`, `time_min`

The km component matrices are long-form with columns:
- horse matrix: `origin_id`, `origin_type`, `dest_id`, `dest_type`, `horse_km`
- rail matrix: `origin_id`, `origin_type`, `dest_id`, `dest_type`, `rail_km`

In the *new* approach we first collapse point-to-point travel times into a **district-to-district** matrix.
We also collapse horse and rail km separately into district-to-district weighted means.

### Point -> district membership and weights
We treat each district as having these **points**:
- the district centroid point `District:<District>` weighted by **rural population**
- each city point `City:<City>` (assigned to a district) weighted by **city population**

For each year, we build a point table with:
- `point_id`
- `District` (the district the point belongs to)
- `pop` (the point's population weight in that year)

### District-to-district weighted distances
For every ordered pair of districts \(A,B\), \(A\neq B\), we compute a **population-pair weighted mean** of distances between all points \(p\in A\) and \(q\in B\):

\[
d_{A\to B} = \frac{\sum_{p\in A}\sum_{q\in B} (Pop_p\cdot Pop_q)\, d_{p\to q}}
{\sum_{p\in A}\sum_{q\in B} (Pop_p\cdot Pop_q)}
\]

This is computed for:
- `time_min`
- `horse_km`
- `rail_km`
- `total_km = horse_km + rail_km`

Pairs with missing/zero population get weight 0 and do not affect the mean.

Once we have district-to-district times, we compute **district market access** using **total district population** (rural + all cities).


In [ ]:
def build_point_population_table_for_year(year: str) -> pd.DataFrame:
    """Point-level table with point_id, District membership, and population weight."""
    # District (rural) points
    d = rural_keep[["District", year]].copy()
    d["point_id"] = "District:" + d["District"].astype(str)
    d["point_type"] = "District"
    d.rename(columns={year: "pop"}, inplace=True)

    # City points (belong to a district)
    c = city_keep[["City", "District", year]].copy()
    c["point_id"] = "City:" + c["City"].astype(str)
    c["point_type"] = "City"
    c.rename(columns={year: "pop"}, inplace=True)

    pop = pd.concat(
        [
            d[["point_id", "point_type", "District", "pop"]],
            c[["point_id", "point_type", "District", "pop"]],
        ],
        ignore_index=True,
    )

    pop["pop"] = pd.to_numeric(pop["pop"], errors="coerce").fillna(0.0)
    return pop


def build_district_population_table_for_year(year: str) -> pd.DataFrame:
    """District total population = rural + sum of city populations (for that year)."""
    rural_y = rural_keep[["District", year]].copy().rename(columns={year: "rural_pop"})
    rural_y["rural_pop"] = pd.to_numeric(rural_y["rural_pop"], errors="coerce").fillna(0.0)

    city_y = city_keep[["District", year]].copy().rename(columns={year: "city_pop"})
    city_y["city_pop"] = pd.to_numeric(city_y["city_pop"], errors="coerce").fillna(0.0)

    city_ag = city_y.groupby("District", as_index=False)["city_pop"].sum().rename(columns={"city_pop": "city_pop_sum"})

    out = rural_y.merge(city_ag, on="District", how="left")
    out["city_pop_sum"] = out["city_pop_sum"].fillna(0.0)
    out["pop_total"] = out["rural_pop"] + out["city_pop_sum"]
    return out


def _attach_district_weights(dm: pd.DataFrame, pop_points: pd.DataFrame) -> pd.DataFrame:
    """Attach origin/destination district membership and population weights."""
    pp = pop_points[["point_id", "District", "pop"]].copy()

    dm2 = dm.merge(pp, left_on="origin_id", right_on="point_id", how="left").rename(
        columns={"District": "origin_district", "pop": "origin_pop"}
    )
    dm2 = dm2.merge(pp, left_on="dest_id", right_on="point_id", how="left", suffixes=("", "_dest")).rename(
        columns={"District": "dest_district", "pop": "dest_pop"}
    )

    dm2 = dm2.dropna(subset=["origin_district", "dest_district"]).copy()
    dm2 = dm2[dm2["origin_district"] != dm2["dest_district"]].copy()
    dm2["w"] = dm2["origin_pop"].fillna(0.0) * dm2["dest_pop"].fillna(0.0)
    dm2 = dm2[dm2["w"] > 0].copy()
    return dm2


def compute_district_distance_matrix(dm: pd.DataFrame, pop_points: pd.DataFrame) -> pd.DataFrame:
    """Collapse point-to-point time matrix into district-to-district weighted means."""
    needed = {"origin_id", "dest_id", "time_min"}
    missing = needed - set(dm.columns)
    if missing:
        raise RuntimeError(f"Distance matrix is missing columns: {sorted(missing)}")

    dm2 = _attach_district_weights(dm, pop_points)
    dm2["time_min"] = pd.to_numeric(dm2["time_min"], errors="coerce")
    dm2 = dm2[dm2["time_min"].notna()].copy()

    grp = dm2.groupby(["origin_district", "dest_district"], as_index=False).agg(
        w_sum=("w", "sum"),
        wt_sum=("time_min", lambda s: float(np.sum(s * dm2.loc[s.index, "w"]))),
    )
    grp["time_min"] = grp["wt_sum"] / grp["w_sum"]
    return grp[["origin_district", "dest_district", "time_min"]].copy()


def compute_district_distance_matrix_multi(
    dm: pd.DataFrame,
    pop_points: pd.DataFrame,
    metric_cols: list[str],
) -> pd.DataFrame:
    """Collapse point-level metrics into district-to-district weighted means for each metric."""
    needed = {"origin_id", "dest_id", *metric_cols}
    missing = needed - set(dm.columns)
    if missing:
        raise RuntimeError(f"Distance matrix is missing columns: {sorted(missing)}")

    dm2 = _attach_district_weights(dm, pop_points)

    for col in metric_cols:
        dm2[col] = pd.to_numeric(dm2[col], errors="coerce")
        valid = dm2[col].notna() & np.isfinite(dm2[col])
        dm2[f"{col}__wt"] = np.where(valid, dm2[col] * dm2["w"], 0.0)
        dm2[f"{col}__w"] = np.where(valid, dm2["w"], 0.0)

    agg_dict = {}
    for col in metric_cols:
        agg_dict[f"{col}__wt"] = "sum"
        agg_dict[f"{col}__w"] = "sum"

    grp = dm2.groupby(["origin_district", "dest_district"], as_index=False).agg(agg_dict)

    out = grp[["origin_district", "dest_district"]].copy()
    for col in metric_cols:
        denom = grp[f"{col}__w"]
        out[col] = np.where(denom > 0, grp[f"{col}__wt"] / denom, np.nan)

    return out


def load_dm(path: Path) -> pd.DataFrame:
    dm = pd.read_csv(path)
    needed = {"origin_id", "origin_type", "dest_id", "dest_type", "time_min"}
    missing = needed - set(dm.columns)
    if missing:
        raise RuntimeError(f"{path} is missing columns: {sorted(missing)}")
    return dm


def load_dm_component(path: Path, metric_col: str) -> pd.DataFrame:
    dm = pd.read_csv(path)
    needed = {"origin_id", "origin_type", "dest_id", "dest_type", metric_col}
    missing = needed - set(dm.columns)
    if missing:
        raise RuntimeError(f"{path} is missing columns: {sorted(missing)}")
    return dm[["origin_id", "origin_type", "dest_id", "dest_type", metric_col]].copy()


def load_km_components(horse_path: Path, rail_path: Path) -> pd.DataFrame:
    horse = load_dm_component(horse_path, "horse_km")
    rail = load_dm_component(rail_path, "rail_km")

    keys = ["origin_id", "origin_type", "dest_id", "dest_type"]
    km = horse.merge(rail, on=keys, how="outer")

    km["horse_km"] = pd.to_numeric(km["horse_km"], errors="coerce")
    km["rail_km"] = pd.to_numeric(km["rail_km"], errors="coerce")
    km["total_km"] = km["horse_km"] + km["rail_km"]

    missing_horse = km["horse_km"].isna().sum()
    missing_rail = km["rail_km"].isna().sum()
    if missing_horse or missing_rail:
        print(f"Warning: km matrix merge has missing components: horse={missing_horse}, rail={missing_rail}")

    return km


# Load time matrices (baseline)
dm_1913 = load_dm(DM_1913)
dm_1924 = load_dm(DM_1924)
dm_1938 = load_dm(DM_1938)

# Load km component matrices (baseline)
dm_km_1913 = load_km_components(DM_HORSE_KM_1913, DM_RAIL_KM_1913)
dm_km_1924 = load_km_components(DM_HORSE_KM_1924, DM_RAIL_KM_1924)
dm_km_1938 = load_km_components(DM_HORSE_KM_1938, DM_RAIL_KM_1938)

print("Time DM sizes:", len(dm_1913), len(dm_1924), len(dm_1938))
print("KM DM sizes:", len(dm_km_1913), len(dm_km_1924), len(dm_km_1938))
dm_1913.head()


DM sizes: 829921 829921 829921


,origin_id,origin_type,dest_id,dest_type,time_min
0,District:AUGUSTOWSKI,District,District:AUGUSTOWSKI,District,0.000000
1,District:AUGUSTOWSKI,District,District:BARANOWICKI,District,1618.424981
2,District:AUGUSTOWSKI,District,District:BIALSKI (BIAŁA KRAKOWSKA),District,3832.500705
3,District:AUGUSTOWSKI,District,District:BIALSKI (BIAŁA PODLASKA),District,1863.051221
4,District:AUGUSTOWSKI,District,District:BIAŁOSTOCKI,District,1073.325185


In [ ]:
# Compute district-to-district distance matrices for each year
results_dist = {}
results_dist_km = {}

for year, dm_time, dm_km in [
    ("1913", dm_1913, dm_km_1913),
    ("1924", dm_1924, dm_km_1924),
    ("1938", dm_1938, dm_km_1938),
]:
    print(f"--- Computing district distance matrices for {year} ---")
    pop_points = build_point_population_table_for_year(year)

    ddm_time = compute_district_distance_matrix(dm_time, pop_points)
    ddm_km = compute_district_distance_matrix_multi(
        dm_km,
        pop_points,
        metric_cols=["horse_km", "rail_km", "total_km"],
    )

    results_dist[year] = ddm_time
    results_dist_km[year] = ddm_km

    print(f"Time district-pairs computed: {len(ddm_time):,}")
    print(f"KM district-pairs computed: {len(ddm_km):,}")

results_dist["1924"].head()


--- Computing district distance matrix for 1913 ---
District-pairs computed for 60,762 (ordered) district pairs
--- Computing district distance matrix for 1924 ---
District-pairs computed for 60,762 (ordered) district pairs
--- Computing district distance matrix for 1939 ---
District-pairs computed for 60,762 (ordered) district pairs


,origin_district,dest_district,time_min
0,AUGUSTOWSKI,BARANOWICKI,1688.451193
1,AUGUSTOWSKI,BIALSKI (BIAŁA KRAKOWSKA),3489.558583
2,AUGUSTOWSKI,BIALSKI (BIAŁA PODLASKA),1698.700207
3,AUGUSTOWSKI,BIAŁOSTOCKI,949.837030
4,AUGUSTOWSKI,BIELSKI (BIELSK PODLASKI),1210.176598


## 4) Compute district-level market access from district distances (using total district population)

For each year we now have a district-to-district travel-time matrix \(t_{i\to j}\).

We compute market access for each **origin district** \(i\) using the **total population** of destination districts \(j\):

- **Power decay** (default):  \( MA_i = \sum_{j\neq i} \frac{Pop_j}{(t_{i\to j}+\varepsilon)^\theta} \)
- **Exponential decay** (optional): \( MA_i = \sum_{j\neq i} Pop_j\, e^{-\alpha t_{i\to j}} \)

The output table contains one row per district per year with:
- `rural_pop`, `city_pop_sum`, `pop_total`
- `MA_district`


In [ ]:
def compute_district_market_access(year: str, ddm: pd.DataFrame) -> pd.DataFrame:
    # District total population table
    pop_d = build_district_population_table_for_year(year).copy()
    pop_map = dict(zip(pop_d["District"], pop_d["pop_total"]))

    dm2 = ddm.copy()
    dm2["dest_pop_total"] = dm2["dest_district"].map(pop_map).fillna(0.0)

    # Exclude self-pairs if any slipped in
    dm2 = dm2[dm2["origin_district"] != dm2["dest_district"]].copy()

    t = pd.to_numeric(dm2["time_min"], errors="coerce").fillna(np.inf)

    if USE_EXPONENTIAL:
        dm2["contrib"] = dm2["dest_pop_total"] * np.exp(-ALPHA_PER_MIN * t)
    else:
        dm2["contrib"] = dm2["dest_pop_total"] / np.power(t + EPS_MIN, THETA)

    ma = dm2.groupby("origin_district", as_index=False)["contrib"].sum().rename(
        columns={"origin_district": "District", "contrib": "MA_district"}
    )

    out = pop_d.merge(ma, on="District", how="left")
    out["MA_district"] = out["MA_district"].fillna(0.0)

    out["year"] = int(year)
    return out[["District", "year", "rural_pop", "city_pop_sum", "pop_total", "MA_district"]].copy()


district_ma_frames = []
for year in ["1913", "1924", "1938"]:
    district_ma = compute_district_market_access(year, results_dist[year])
    district_ma_frames.append(district_ma)

district_ma_all = pd.concat(district_ma_frames, ignore_index=True)

district_ma_all.head()


,District,year,rural_pop,city_pop_sum,pop_total,MA_district
0,KOŚCIAŃSKI,1913,66176.8,15539.2,81716.0,11969.201690
1,ZBOROWSKI,1913,51189.6,9343.4,60533.0,9959.150601
2,MIĘDZYCHODZKI,1913,23670.2,5740.0,29410.2,9232.801872
3,KOŁOMYJSKI,1913,107218.4,35661.6,142880.0,10342.333524
4,DZIŚNIEŃSKI,1913,114329.2,8986.2,123315.4,5880.722270


## 5) Save market access by district and compute changes

We save:
- `market_access_by_district.csv` with MA for 1913, 1924, 1938

And compute changes for:
- 1913 → 1924
- 1924 → 1938
- 1913 → 1938

We provide three change measures:
- `delta` = MA₂ − MA₁
- `pct_change` = (MA₂ / MA₁) − 1
- `log_change` = log(MA₂) − log(MA₁) (safe with small epsilon)

Each change is saved to its own CSV.



In [ ]:
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

# Save district-to-district distance matrices
dist_dir = OUT_DIR / "distance_matrices"
dist_dir.mkdir(exist_ok=True)

for year, dm_dist in results_dist.items():
    path = dist_dir / f"district_distance_matrix_{year}.csv"
    dm_dist.to_csv(path, index=False)
    print("Wrote:", path)

for year, dm_dist_km in results_dist_km.items():
    path = dist_dir / f"district_distance_matrix_km_{year}_baseline.csv"
    dm_dist_km.to_csv(path, index=False)
    print("Wrote:", path)

# Save market access levels
levels_path = OUT_DIR / "market_access_by_district.csv"
district_ma_all.to_csv(levels_path, index=False)
print("Wrote:", levels_path)

def compute_changes(df_levels: pd.DataFrame, y0: int, y1: int) -> pd.DataFrame:
    a = df_levels[df_levels["year"] == y0][["District", "MA_district"]].rename(columns={"MA_district": f"MA_{y0}"})
    b = df_levels[df_levels["year"] == y1][["District", "MA_district"]].rename(columns={"MA_district": f"MA_{y1}"})
    m = a.merge(b, on="District", how="outer")
    eps = 1e-12

    m["delta"] = m[f"MA_{y1}"] - m[f"MA_{y0}"]
    m["pct_change"] = (m[f"MA_{y1}"] / (m[f"MA_{y0}"] + eps)) - 1.0
    m["log_change"] = np.log(m[f"MA_{y1}"] + eps) - np.log(m[f"MA_{y0}"] + eps)
    m["from_year"] = y0
    m["to_year"] = y1
    return m

chg_1913_1924 = compute_changes(district_ma_all, 1913, 1924)
chg_1924_1938 = compute_changes(district_ma_all, 1924, 1938)
chg_1913_1938 = compute_changes(district_ma_all, 1913, 1938)

p1 = OUT_DIR / "market_access_change_1913_1924.csv"
p2 = OUT_DIR / "market_access_change_1924_1938.csv"
p3 = OUT_DIR / "market_access_change_1913_1938.csv"

chg_1913_1924.to_csv(p1, index=False)
chg_1924_1938.to_csv(p2, index=False)
chg_1913_1938.to_csv(p3, index=False)

print("Wrote:", p1)
print("Wrote:", p2)
print("Wrote:", p3)

chg_1913_1924.head()


Wrote: outputs\distance_matrices\district_distance_matrix_1913.csv
Wrote: outputs\distance_matrices\district_distance_matrix_1924.csv
Wrote: outputs\distance_matrices\district_distance_matrix_1939.csv
Wrote: outputs\market_access_by_district.csv
Wrote: outputs\market_access_change_1913_1924.csv
Wrote: outputs\market_access_change_1924_1939.csv
Wrote: outputs\market_access_change_1913_1939.csv


,District,MA_1913,MA_1924,delta,pct_change,log_change,from_year,to_year
0,AUGUSTOWSKI,8235.934811,11968.310973,3732.376162,0.453182,0.373756,1913,1924
1,BARANOWICKI,9113.290639,11910.362113,2797.071474,0.306922,0.267675,1913,1924
2,BIALSKI (BIAŁA KRAKOWSKA),14041.982748,18423.532612,4381.549864,0.312032,0.271577,1913,1924
3,BIALSKI (BIAŁA PODLASKA),12071.273667,18214.823632,6143.549965,0.508940,0.411407,1913,1924
4,BIAŁOSTOCKI,11185.358748,16461.791494,5276.432746,0.471727,0.386436,1913,1924


## 6) Quick sanity checks

- Does every district have MA values?
- Are any values missing or infinite?
- What does the distribution look like?



In [9]:
# Coverage by year
cov = district_ma_all.pivot_table(index="year", values="MA_district", aggfunc=["count", "min", "median", "max"])
cov


,count,min,median,max
,MA_district,MA_district,MA_district,MA_district
year,,,,
1913,247,4968.660540,11429.856282,20922.098654
1924,247,8589.765376,16449.806468,27671.681518
1939,247,13366.108070,25640.694599,51442.012804


In [ ]:
# --- Helper: stable legend bounds for non-negative variables ---
def robust_positive_bounds(series: pd.Series, qmax=0.98):
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return 0.0, None
    vmax = float(s.quantile(qmax))
    return 0.0, vmax


# --- Plot market access levels ---
ma_levels = district_ma_all.copy()
ma_levels["District"] = ma_levels["District"].astype(str).str.strip()

# MA is non-negative: force legend to start at 0 to avoid negative ticks
vmin, vmax = robust_positive_bounds(ma_levels["MA_district"])

for y in [1913, 1924, 1938]:
    df_y = (
        ma_levels.loc[ma_levels["year"] == y, ["District", "MA_district"]]
        .set_index("District")
    )

    adm_history_plotter.plot_dataset(
        df=df_y,
        col_name="MA_district",
        adm_level="District",
        adm_state_date=ADM_STATE_DATE,
        save_to_path=str(MA_PLOTS_DIR / f"market_access_{y}.png"),
        title=f"Market access by district ({y})",
        legend_min=vmin,   # always 0.0
        legend_max=vmax,
        cmap="OrRd",
        custom_grouping=d_city_mapping
    )

In [ ]:
# --- Plot market access changes ---
change_specs = [
    ("1913_1924", chg_1913_1924),
    ("1924_1938", chg_1924_1938),
    ("1913_1938", chg_1913_1938),
]

for tag, df in change_specs:
    dfx = df.copy()
    dfx["District"] = dfx["District"].astype(str).str.strip()
    dfx = dfx.set_index("District")

    for col, cmap, title in [
        ("delta", "OrRd", f"Market access change Δ ({tag})"),
        ("pct_change", "OrRd", f"Market access % change ({tag})"),
        ("log_change", "OrRd", f"Market access log change ({tag})"),
    ]:
        # MA changes are non-negative here: force legend to start at 0
        vmin, vmax = robust_positive_bounds(dfx[col])

        adm_history_plotter.plot_dataset(
            df=dfx[[col]],
            col_name=col,
            adm_level="District",
            adm_state_date=ADM_STATE_DATE,
            save_to_path=str(MA_PLOTS_DIR / f"market_access_change_{col}_{tag}.png"),
            title=title,
            legend_min=vmin,   # always 0.0
            legend_max=vmax,
            cmap=cmap,
            custom_grouping=d_city_mapping
        )

E:\git_projects\border_harmonization_toolkit\src\administrative_history\core\plotter.py:249: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(10, 8))


## 7) Population-weighted distance to selected border crossings

In this step we compute, for each district and each year (1913, 1924, 1938), the **population-weighted travel time** from the district to three border-crossing “gates”:

- **Makoszowy–Gliwice**
- **Zbąszyń–Zbąszynek**
- **Gdańsk**, defined as the **minimum** travel time to any of:
  - Koźliny (Kozliny)
  - Kolibki
  - Sulmin

The distance for a district is defined analogously to our district market-access aggregation: we treat the district as a mixture of population points:

- the **district centroid** (weight = **rural population**)
- all **cities located in the district** (weight = **city population**)

For each point (district centroid or city), we read its travel time to the border crossing from the **distance matrix**.  
For the Gdańsk gate we take the **minimum** across the three Gdańsk crossings for each point.  
Finally, for each district we compute the **population-weighted mean** distance:

\[
D^{district}_{i}=\frac{R_i \cdot d_{centroid,i} + \sum_{c \in i} Pop_c \cdot d_{city,c}}{R_i + \sum_{c \in i} Pop_c}
\]

We output:
- a combined table `district_border_distance_by_year.csv`
- and separate year tables if needed.


In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

# Border crossing point IDs used in the distance matrices (must match how you named them earlier)
BORDER_MAKOSZOWY = "Border_Crossing:Makoszowy-Gliwice"
BORDER_ZBASZYN   = "Border_Crossing:Zbaszyn-Zbaszynek"

# For Gdańsk gate: take min distance to any of these three
BORDER_GDANSK_SET = [
    "Border_Crossing:Kozliny",   # (Koźliny)
    "Border_Crossing:Kolibki",
    "Border_Crossing:Sulmin",
]

def _dm_point_to_dest_series(dm: pd.DataFrame, dest_id: str) -> pd.Series:
    """
    Returns a Series indexed by origin_id with time_min to a single destination dest_id.
    """
    sub = dm.loc[dm["dest_id"] == dest_id, ["origin_id", "time_min"]].copy()
    sub["time_min"] = pd.to_numeric(sub["time_min"], errors="coerce")
    # If duplicates exist, keep minimum time
    return sub.groupby("origin_id")["time_min"].min()

def _dm_point_to_gdansk_min_series(dm: pd.DataFrame, gdansk_dest_ids: list[str]) -> pd.Series:
    """
    Returns a Series indexed by origin_id with min time_min to any destination in gdansk_dest_ids.
    """
    sub = dm.loc[dm["dest_id"].isin(gdansk_dest_ids), ["origin_id", "dest_id", "time_min"]].copy()
    sub["time_min"] = pd.to_numeric(sub["time_min"], errors="coerce")
    return sub.groupby("origin_id")["time_min"].min()

def _safe_weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    v = pd.to_numeric(values, errors="coerce")
    w = pd.to_numeric(weights, errors="coerce").fillna(0.0)
    mask = v.notna() & np.isfinite(v) & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float(np.sum(v[mask] * w[mask]) / np.sum(w[mask]))


In [14]:
def compute_district_border_distances_for_year(
    year: str,
    dm: pd.DataFrame,
    city_keep: pd.DataFrame,
    rural_keep: pd.DataFrame,
) -> pd.DataFrame:
    """
    Computes population-weighted travel time from each district to:
      - Makoszowy-Gliwice
      - Zbaszyn-Zbaszynek
      - Gdansk gate (min of Kozliny/Kolibki/Sulmin)

    Returns a DataFrame with one row per district.
    """
    # --- Build point IDs + weights ---
    rural_y = rural_keep[["District", year]].copy()
    rural_y["District"] = rural_y["District"].astype(str).str.strip()
    rural_y["point_id"] = "District:" + rural_y["District"]
    rural_y.rename(columns={year: "rural_pop"}, inplace=True)
    rural_y["rural_pop"] = pd.to_numeric(rural_y["rural_pop"], errors="coerce").fillna(0.0)

    city_y = city_keep[["City", "District", year]].copy()
    city_y["City"] = city_y["City"].astype(str).str.strip()
    city_y["District"] = city_y["District"].astype(str).str.strip()
    city_y["point_id"] = "City:" + city_y["City"]
    city_y.rename(columns={year: "city_pop"}, inplace=True)
    city_y["city_pop"] = pd.to_numeric(city_y["city_pop"], errors="coerce").fillna(0.0)

    # --- Distances from each origin point to each border ---
    d_mak = _dm_point_to_dest_series(dm, BORDER_MAKOSZOWY)
    d_zba = _dm_point_to_dest_series(dm, BORDER_ZBASZYN)
    d_gda = _dm_point_to_gdansk_min_series(dm, BORDER_GDANSK_SET)

    # Map point distances onto rural centroid points
    rural_y["d_mak_min"] = rural_y["point_id"].map(d_mak)
    rural_y["d_zba_min"] = rural_y["point_id"].map(d_zba)
    rural_y["d_gda_min"] = rural_y["point_id"].map(d_gda)

    # Map point distances onto city points
    city_y["d_mak_min"] = city_y["point_id"].map(d_mak)
    city_y["d_zba_min"] = city_y["point_id"].map(d_zba)
    city_y["d_gda_min"] = city_y["point_id"].map(d_gda)

    # Aggregate cities by district: weighted mean per border
    def city_weighted(d: pd.DataFrame, col: str) -> float:
        return _safe_weighted_mean(d[col], d["city_pop"])

    city_ag = city_y.groupby("District", as_index=False).apply(
        lambda d: pd.Series({
            "city_pop_sum": float(d["city_pop"].sum()),
            "city_d_mak": city_weighted(d, "d_mak_min"),
            "city_d_zba": city_weighted(d, "d_zba_min"),
            "city_d_gda": city_weighted(d, "d_gda_min"),
        })
    ).reset_index(drop=True)

    # Combine with rural centroid distances, then district-weighted mean
    out = rural_y.merge(city_ag, on="District", how="left")
    out["city_pop_sum"] = out["city_pop_sum"].fillna(0.0)

    out["pop_total"] = out["rural_pop"] + out["city_pop_sum"]

    # District-level weighted mean distances (minutes)
    out["dist_makoszowy_min"] = (
        (out["rural_pop"] * out["d_mak_min"].fillna(np.nan)) +
        (out["city_pop_sum"] * out["city_d_mak"].fillna(np.nan))
    ) / out["pop_total"].replace({0.0: np.nan})

    out["dist_zbaszyn_min"] = (
        (out["rural_pop"] * out["d_zba_min"].fillna(np.nan)) +
        (out["city_pop_sum"] * out["city_d_zba"].fillna(np.nan))
    ) / out["pop_total"].replace({0.0: np.nan})

    out["dist_gdansk_min"] = (
        (out["rural_pop"] * out["d_gda_min"].fillna(np.nan)) +
        (out["city_pop_sum"] * out["city_d_gda"].fillna(np.nan))
    ) / out["pop_total"].replace({0.0: np.nan})

    out["year"] = int(year)

    return out[[
        "District", "year",
        "rural_pop", "city_pop_sum", "pop_total",
        "dist_makoszowy_min", "dist_zbaszyn_min", "dist_gdansk_min"
    ]].copy()


In [ ]:
# Compute for all three years (expects dm_1913, dm_1924, dm_1938 already loaded earlier)
border_dist_1913 = compute_district_border_distances_for_year("1913", dm_1913, city_keep, rural_keep)
border_dist_1924 = compute_district_border_distances_for_year("1924", dm_1924, city_keep, rural_keep)
border_dist_1938 = compute_district_border_distances_for_year("1938", dm_1938, city_keep, rural_keep)

district_border_dist_by_year = pd.concat([border_dist_1913, border_dist_1924, border_dist_1938], ignore_index=True)

display(district_border_dist_by_year.head())

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

out_path = OUT_DIR / "district_border_distance_by_year.csv"
district_border_dist_by_year.to_csv(out_path, index=False)
print("Wrote:", out_path)

# Optional: year-specific files
for y in [1913, 1924, 1938]:
    p = OUT_DIR / f"district_border_distance_{y}.csv"
    district_border_dist_by_year.loc[district_border_dist_by_year["year"] == y].to_csv(p, index=False)
    print("Wrote:", p)


C:\Users\janek\AppData\Local\Temp\ipykernel_11588\697835043.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  city_ag = city_y.groupby("District", as_index=False).apply(
C:\Users\janek\AppData\Local\Temp\ipykernel_11588\697835043.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  city_ag = city_y.groupby("District", as_index=False).apply(
C:\Users\janek\AppData\Local\Temp\ipykernel_11588\697835043.py:48: FutureW

,District,year,rural_pop,city_pop_sum,pop_total,dist_makoszowy_min,dist_zbaszyn_min,dist_gdansk_min
0,KOŚCIAŃSKI,1913,66176.8,15539.2,81716.0,2498.464260,430.019524,1633.708164
1,ZBOROWSKI,1913,51189.6,9343.4,60533.0,2826.800728,5165.391732,5229.876271
2,MIĘDZYCHODZKI,1913,23670.2,5740.0,29410.2,2976.191162,390.793062,1884.245317
3,KOŁOMYJSKI,1913,107218.4,35661.6,142880.0,2931.683073,5270.274076,5547.069393
4,DZIŚNIEŃSKI,1913,114329.2,8986.2,123315.4,4859.023295,5280.120117,4884.168696


Wrote: outputs\district_border_distance_by_year.csv
Wrote: outputs\district_border_distance_1913.csv
Wrote: outputs\district_border_distance_1924.csv
Wrote: outputs\district_border_distance_1939.csv


## 8) Plot border distance levels and changes

We now visualize the population-weighted travel time (minutes) from each district to:
- Makoszowy–Gliwice
- Zbąszyń–Zbąszynek
- Gdańsk gate (min of Koźliny/Kolibki/Sulmin)

We produce choropleths for:
- levels in 1913, 1924, 1938
- changes (Δ minutes) for 1913–1924, 1924–1938, 1913–1938

All plots are saved to `./plots/` using `adm_history_plotter.plot_dataset`.


In [ ]:
def robust_bounds(series: pd.Series, qmin=0.02, qmax=0.98):
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return None, None
    return float(s.quantile(qmin)), float(s.quantile(qmax))


# Columns to plot (minutes)
BORDER_DIST_COLS = [
    ("dist_makoszowy_min", "Distance to Makoszowy–Gliwice (min)"),
    ("dist_zbaszyn_min",   "Distance to Zbąszyń–Zbaszynek (min)"),
    ("dist_gdansk_min",    "Distance to Gdańsk gate (min)"),
]

bd = district_border_dist_by_year.copy()
bd["District"] = bd["District"].astype(str).str.strip()

# Consistent legend across years for each border measure
bounds_by_col = {col: robust_bounds(bd[col]) for col, _ in BORDER_DIST_COLS}

for col, title_stub in BORDER_DIST_COLS:
    # Create subfolder per border measure
    subdir = BORDER_CROSSING_PLOTS_DIR / col
    subdir.mkdir(parents=True, exist_ok=True)

    vmin, vmax = bounds_by_col[col]

    for y in [1913, 1924, 1938]:
        df_y = (
            bd.loc[bd["year"] == y, ["District", col]]
              .set_index("District")
        )

        out_path = subdir / f"{col}_{y}.png"

        adm_history_plotter.plot_dataset(
            df=df_y,
            col_name=col,
            adm_level="District",
            adm_state_date=ADM_STATE_DATE,
            save_to_path=str(out_path),
            title=f"{title_stub} — {y}",
            legend_min=vmin,
            legend_max=vmax,
            cmap="OrRd",
            custom_grouping=d_city_mapping
        )


In [ ]:
def compute_level_change(df_levels: pd.DataFrame, y0: int, y1: int, col: str) -> pd.DataFrame:
    a = df_levels.loc[df_levels["year"] == y0, ["District", col]].rename(columns={col: f"{col}_{y0}"})
    b = df_levels.loc[df_levels["year"] == y1, ["District", col]].rename(columns={col: f"{col}_{y1}"})
    m = a.merge(b, on="District", how="outer")
    m["delta"] = m[f"{col}_{y1}"] - m[f"{col}_{y0}"]
    m["from_year"] = y0
    m["to_year"] = y1
    return m.set_index("District")


change_pairs = [(1913, 1924), (1924, 1938), (1913, 1938)]

for col, title_stub in BORDER_DIST_COLS:
    # Variable-level folder
    var_dir = BORDER_CROSSING_PLOTS_DIR / col / "changes"
    var_dir.mkdir(parents=True, exist_ok=True)

    for y0, y1 in change_pairs:
        ch = compute_level_change(bd, y0, y1, col)

        # Symmetric legend around 0 (robust)
        s = pd.to_numeric(ch["delta"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        vmax = float(np.nanquantile(np.abs(s), 0.98)) if np.isfinite(s).any() else None
        vmin = -vmax if vmax is not None else None

        out_path = var_dir / f"{col}_change_{y0}_{y1}.png"

        adm_history_plotter.plot_dataset(
            df=ch[["delta"]],
            col_name="delta",
            adm_level="District",
            adm_state_date=ADM_STATE_DATE,
            save_to_path=str(out_path),
            title=f"{title_stub} — change (Δ min) {y0}→{y1}",
            legend_min=vmin,
            legend_max=vmax,
            cmap="RdBu",
            custom_grouping=d_city_mapping
        )
